# 08. Consumption Tables: the tables that will power the dashboard

**Project:** Buffer Delay Allocation Engine (Analytics Capstone Project, Team T19)
**Team:** Vaibhav Maheshwari (BA05057), Prashansa Naomi Konduru (BA05042), Anil Kumar (BA05008)
**Faculty:** Mr. Nikhil D Jonathan
**Data:** `aa_clean_v1.0.parquet` (tag `data-v1.0`): 1,641,071 American Airlines mainline flights, 1 Jan 2025 to 31 Aug 2026

**What this notebook does.** Notebooks 04 to 06 answered the research questions. This notebook turns their results into **clean, ready-to-use tables** (often called
*gold* or *consumption* tables) that a dashboard can read directly, without doing any analysis itself. It:

1. builds the turnaround table, the pass-through curve, the prediction models and the buffer engine again, with the same rules and settings as notebooks 04 to 06;
2. stores the **actual outputs of the models and of the optimisation engine**, one row per turnaround, with IDs, reasons and version labels;
3. calculates every **KPI** the dashboard will show;
4. checks the tables (row counts, no missing keys, budgets respected, headline numbers matching notebooks 04 to 06);
5. saves them: always as **parquet files** (and CSV for the smaller tables), and, when it runs in **Databricks**, also as **Delta tables** in `workspace.bufferengine`.

It runs on its own. It needs only `aa_clean_v1.0.parquet`.

## How to run this notebook

You need one data file, **`aa_clean_v1.0.parquet`** (68 MB, the cleaned flight data from D2), and these Python packages: pandas, numpy, matplotlib, pyarrow, scikit-learn, lightgbm.
It takes about 3 to 6 minutes and needs about 4 GB of free memory (a normal laptop or free Google Colab is enough). Then choose *Restart and Run All*.

| Where you run it | What to do |
|---|---|
| **Your laptop** (Jupyter, VS Code, Anaconda) | Put `aa_clean_v1.0.parquet` in the same folder as this notebook (or in a folder called `data`). If a package is missing, run `pip install pandas numpy matplotlib pyarrow scikit-learn lightgbm` once. |
| **Google Colab** | Upload this notebook (File, then Upload notebook). Run the first code cell. When it asks, click *Choose files* and select `aa_clean_v1.0.parquet` (or upload the file to the Colab file panel first). Nothing needs to be installed. |

The notebook checks the data file against its published checksum, so you know it is the exact dataset from D2. Charts and tables are saved in a
folder called `modelling_outputs` next to the notebook (or `docs/modelling` if it is run inside the project repository).

**In Databricks.** Import the notebook, attach it to a cluster or Serverless compute, and *Run all*. The data file is found automatically in the Volume
`/Volumes/workspace/default/bufferengine/processed/`. If `lightgbm` is missing, run `%pip install lightgbm` in a cell first. The tables are written to the schema set in the
settings cell below (default `workspace.bufferengine`), and the files to `/Volumes/workspace/default/bufferengine/consumption/`.

## The tables this notebook creates

| Group | Table | One row per | Dashboard page |
|---|---|---|---|
| Lookup | `gold_dim_airport` | airport | filters, pages 3, 4, 6 |
| Lookup | `gold_dim_date` | day | filters, all pages |
| Headline | `gold_kpi_headline` | KPI | 1 Executive summary |
| Network | `gold_network_daily` | day | 2 Network delay health |
| Airports | `gold_airport_month` | airport x month | 3 Airport risk |
| Airports | `gold_airport_hour` | airport x hour | 3 Airport risk (heatmap) |
| Propagation | `gold_passthrough_curve` | ground-time band x segment | 4 Delay propagation |
| Propagation | `gold_delay_by_leg` | flight number in the aircraft's day x period | 4 Delay propagation |
| Model output | `gold_ml_predictions` | turnaround (test period) | 5 Risk forecast, 6a Recommendations |
| Model quality | `gold_model_metrics` | model x predictor x period x measure | 5 Risk forecast, 7 Trust |
| Model quality | `gold_model_deciles` | model x risk group | 5 Risk forecast |
| Model quality | `gold_feature_importance` | model x input | 5 Risk forecast |
| Engine output | `gold_buffer_recommendation` | turnaround x budget | 6a Recommendations |
| Engine output | `gold_buffer_plan_daily` | day x budget | 6 Simulator, 6a |
| Engine output | `gold_buffer_plan_airport` | airport x month x budget | 6 Simulator, 6a |
| Simulator | `gold_sim_strategy` | strategy x budget x day type | 1, 6 Simulator |
| Simulator | `gold_sim_daily` | day x strategy x budget | 6 Simulator |
| Simulator | `gold_buffer_allocation` | budget x grouping x group | 6 Simulator |
| Trust | `gold_data_quality` | check | 7 Data and model trust |
| Trust | `gold_engine_run_log` | run | 7 Data and model trust |
| Help | `gold_table_catalog` | table | describes all the tables above |

**Periods.** Descriptive tables (network, airports, propagation) cover all 20 months. Model and engine tables cover the **test period only (Jan to Aug 2026)**, which the
models never saw while learning, so the dashboard never shows a number the model was trained on. We treat the test period as if it were the *upcoming* timetable: the models use timetable
facts only, and the real outcomes are kept next to the predictions as a backtest.

## 1. Settings

In [1]:
# ---- Settings you may change ------------------------------------------------------------------------------------
CATALOG, SCHEMA = "workspace", "bufferengine"      # where the Delta tables go in Databricks
BUDGETS = [2, 5, 10, 15]                            # buffer budgets (average extra minutes per turnaround) to pre-compute
MAIN_BUDGET = 5                                     # the budget used for headline numbers
CSV_MAX_ROWS = 300_000                              # tables bigger than this are saved as parquet only (CSV would be too large)
MODEL_VERSION, ENGINE_VERSION, CURVE_VERSION = "lgbm-1.0", "engine-1.0", "curve-train2025-1.0"
DATA_TAG = "data-v1.0"

## 2. Setup and data check

In [2]:
import sys, hashlib, json, warnings
from pathlib import Path

# 1. Check that the packages we need are installed (on Google Colab they already are).
PKGS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "lightgbm": "lightgbm"}          # import name -> pip name
missing = []
for module, pip_name in PKGS.items():
    try:
        __import__(module)
    except ImportError:
        missing.append(pip_name)
if missing:
    raise ImportError("Please install the missing packages, then run this cell again:\n    pip install " + " ".join(missing))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.labelsize": 10,
                     "legend.frameon": False, "figure.autolayout": True})
BLUE, ORANGE, RED, GREEN, GREY, LIGHT = "#1F3A5F", "#E08A1E", "#C0392B", "#2E8B57", "#8A94A6", "#C9D3E3"
SPLITS = ["train", "validate", "test"]
SEED = 42
RESULTS = {}          # every number we quote is stored here and saved to a .json file at the end
thousands = FuncFormatter(lambda v, pos: f"{v:,.0f}")

# 2. Find the data file: next to the notebook, in 'data' folders, in the Colab file area, or in Downloads.
FILENAME = "aa_clean_v1.0.parquet"
EXPECTED_SHA256 = "3a18a9eafffa77ae74db70579be84255f8b04978efdcdb84524bd2900297f14a"   # published in data/CHECKSUMS.sha256
IN_COLAB = "google.colab" in sys.modules

def locate():
    here = Path.cwd().resolve()
    places = []
    for folder in [here, *here.parents]:
        places += [folder, folder / "data", folder / "data" / "processed"]
    places += [Path("/content"), Path("/content/data"), Path("/content/drive/MyDrive"),
               Path.home() / "Downloads", Path.home() / "Desktop",
               Path("/Volumes/workspace/default/bufferengine/processed")]          # Databricks Volume used by this project
    places += sorted(Path("/Volumes").glob("*/*/*/processed")) if Path("/Volumes").exists() else []   # any other Databricks Volume
    for folder in places:
        if (folder / FILENAME).exists():
            return folder / FILENAME
    return None

PARQUET = locate()
if PARQUET is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose the file {FILENAME} from your computer (68 MB). The upload takes about a minute.")
    files.upload()
    PARQUET = locate()
if PARQUET is None:
    raise FileNotFoundError(f"Could not find {FILENAME}. Put it in the same folder as this notebook (or in a folder called 'data'), "
                            "then run again. On Google Colab, run this cell again and choose the file when asked.")

# 3. Where to save charts and tables.
in_repo = PARQUET.parent.name == "processed" and PARQUET.parent.parent.name == "data" and (PARQUET.parent.parent.parent / "docs").exists()
OUT = (PARQUET.parent.parent.parent / "docs" / "modelling") if in_repo else Path.cwd() / "modelling_outputs"
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    plt.show()

def say(text):
    display(Markdown(text))

def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o

def write_results(filename):
    (OUT / filename).write_text(json.dumps(_clean(RESULTS), indent=1))
    print(f"Saved {filename} and {len(list(FIG.glob('*.png')))} chart files in the folder '{OUT.name}'.")

# 4. Make sure this is exactly the published dataset.
actual = hashlib.sha256(PARQUET.read_bytes()).hexdigest()
print("Data file          :", PARQUET.name)
print("Python / pandas    :", sys.version.split()[0], "/", pd.__version__)
print("Checksum matches D2:", actual == EXPECTED_SHA256)
assert actual == EXPECTED_SHA256, "This file is not the published aa_clean_v1.0.parquet (tag data-v1.0). Please download it again."

Data file          : aa_clean_v1.0.parquet
Python / pandas    : 3.13.16 / 3.1.0
Checksum matches D2: True


In [3]:
import uuid, platform
# Where the files go: next to the data in a Databricks Volume, otherwise a folder next to this notebook
if str(PARQUET).startswith("/Volumes/"):
    OUT = PARQUET.parent.parent / "consumption"
else:
    OUT = Path.cwd() / "consumption_tables"
PQ_DIR, CSV_DIR = OUT / "parquet", OUT / "csv"
PQ_DIR.mkdir(parents=True, exist_ok=True); CSV_DIR.mkdir(parents=True, exist_ok=True)

# Spark is there only in Databricks (or a local Spark set-up); without it we still save the files
spark = globals().get("spark")
if spark is None:
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.getActiveSession()
    except Exception:
        spark = None
WRITE_DELTA = spark is not None
RUN_TS = pd.Timestamp.now(tz="UTC").floor("s")
RUN_ID = f"run-{RUN_TS:%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"
ENV = "Databricks" if "DATABRICKS_RUNTIME_VERSION" in __import__("os").environ else ("Google Colab" if IN_COLAB else "Local / Jupyter")
print("Run ID            :", RUN_ID)
print("Environment       :", ENV)
print("Files saved in    :", OUT.name, "(parquet and csv sub-folders)")
print("Delta tables      :", f"{CATALOG}.{SCHEMA}.gold_*" if WRITE_DELTA else "not written (no Spark session found; files only)")
TABLES = {}   # name -> pandas table, filled as we go

Run ID            : run-20261025-045640-99f821
Environment       : Local / Jupyter
Files saved in    : consumption_tables (parquet and csv sub-folders)
Delta tables      : not written (no Spark session found; files only)


## 3. Flight-level tables (all 20 months)

These tables describe the operation, one flight at a time, before any modelling: how many flights there were, how many left late, how many were cancelled.

**Definitions used.**
* **Operated flight:** a flight that was not cancelled.
* **On-time departure:** an operated flight that left less than 15 minutes late (the US government definition).
* **Late arrival:** an operated flight that arrived 15 or more minutes late (`ArrDel15 = 1`).
* **Disruption day:** a day when more than 10% of scheduled flights were cancelled.

In [4]:
COLS8 = ["FlightDate", "Year", "Month", "DayOfWeek", "Tail_Number", "Flight_Number_Reporting_Airline", "Origin", "OriginCityName", "OriginState",
         "Dest", "DestCityName", "DestState", "Distance", "crs_dep_min", "crs_arr_min", "DepDelay", "DepDel15", "ArrDelay", "ArrDel15",
         "Cancelled", "Diverted", "is_operated", "crs_dep_utc", "crs_arr_utc", "split", "flight_id",
         "dq_missing_tail", "dq_extreme_delay_24h", "dq_time_conflict", "dq_duplicate_key", "dq_outlier_other"]
df = pd.read_parquet(PARQUET, columns=COLS8)
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["hr"] = (df.crs_dep_min // 60).clip(0, 23)
op = df[df.is_operated]
print(f"{len(df):,} scheduled flights, {len(op):,} operated, {df.FlightDate.min().date()} to {df.FlightDate.max().date()}")

daily_cancel = df.groupby("FlightDate").Cancelled.mean()
STORM = set(daily_cancel[daily_cancel > 0.10].index)
TOP10 = df.Origin.value_counts().head(10).index.tolist()

# ---- gold_dim_date
dd = df.groupby("FlightDate").agg(year=("Year", "first"), month=("Month", "first"), weekday=("DayOfWeek", "first"), split=("split", "first"),
                                  scheduled_flights=("flight_id", "size"), cancel_rate=("Cancelled", "mean")).reset_index().rename(columns={"FlightDate": "flight_date"})
dd["is_disruption_day"] = dd.flight_date.isin(STORM)
dd["year_month"] = dd.flight_date.dt.strftime("%Y-%m")
dd["weekday_name"] = dd.flight_date.dt.day_name()
TABLES["gold_dim_date"] = dd[["flight_date", "year", "month", "year_month", "weekday", "weekday_name", "split", "scheduled_flights", "cancel_rate", "is_disruption_day"]]

# ---- network by day (turnaround columns are added after the turnaround table is built)
net = df.groupby("FlightDate").agg(scheduled_flights=("flight_id", "size"), cancelled_flights=("Cancelled", "sum"), diverted_flights=("Diverted", "sum")).reset_index()
opd = op.groupby("FlightDate").agg(operated_flights=("flight_id", "size"), dep_late15_rate=("DepDel15", "mean"), arr_late15_rate=("ArrDel15", "mean"),
                                   avg_dep_delay_min=("DepDelay", "mean"), avg_arr_delay_min=("ArrDelay", "mean"),
                                   p90_arr_delay_min=("ArrDelay", lambda x: x.quantile(0.9))).reset_index()
net = net.merge(opd, on="FlightDate", how="left")
net["cancel_rate"] = net.cancelled_flights / net.scheduled_flights
net["dep_ontime_rate"] = 1 - net.dep_late15_rate

# ---- airport x hour
ah = op.groupby(["Origin", "hr"]).agg(departures=("flight_id", "size"), dep_late15_rate=("DepDel15", "mean"), avg_dep_delay_min=("DepDelay", "mean")).reset_index()
ah = ah.rename(columns={"Origin": "airport", "hr": "dep_hour"})
TABLES["gold_airport_hour"] = ah

# ---- airport x month (flight part)
df["year_month"] = df.FlightDate.dt.strftime("%Y-%m"); op = df[df.is_operated]
am_dep = df.groupby(["Origin", "year_month"]).agg(scheduled_departures=("flight_id", "size"), cancel_rate=("Cancelled", "mean")).reset_index()
am_op = op.groupby(["Origin", "year_month"]).agg(dep_late15_rate=("DepDel15", "mean"), avg_dep_delay_min=("DepDelay", "mean")).reset_index()
am_arr = op.groupby(["Dest", "year_month"]).agg(arrivals=("flight_id", "size"), arr_late15_rate=("ArrDel15", "mean"), avg_arr_delay_min=("ArrDelay", "mean")).reset_index().rename(columns={"Dest": "Origin"})
am = am_dep.merge(am_op, on=["Origin", "year_month"], how="left").merge(am_arr, on=["Origin", "year_month"], how="outer").rename(columns={"Origin": "airport"})

# ---- delay by flight number in the aircraft's day (scheduled order, counting every scheduled flight)
s_ = df[df.Tail_Number.notna()].sort_values(["Tail_Number", "crs_dep_utc"])
df.loc[s_.index, "legno"] = s_.groupby(["Tail_Number", "FlightDate"]).cumcount().values + 1
op = df[df.is_operated & df.legno.notna()].copy()
op["leg"] = op.legno.clip(upper=6).astype(int)
leg = pd.concat([op.assign(period="all"), op.assign(period=op.split)]).groupby(["period", "leg"]).agg(
    flights=("flight_id", "size"), avg_dep_delay_min=("DepDelay", "mean"), median_dep_delay_min=("DepDelay", "median"), dep_late15_rate=("DepDel15", "mean")).reset_index()
leg["leg_label"] = leg.leg.map(lambda v: "6+" if v == 6 else str(v))
TABLES["gold_delay_by_leg"] = leg[["period", "leg", "leg_label", "flights", "avg_dep_delay_min", "median_dep_delay_min", "dep_late15_rate"]]

# ---- airport lookup (flight part)
names = df.groupby("Origin").agg(city=("OriginCityName", "first"), state=("OriginState", "first"), departures=("flight_id", "size")).reset_index()
arrs = df.groupby("Dest").agg(arrivals=("flight_id", "size")).reset_index().rename(columns={"Dest": "Origin"})
dim_ap = names.merge(arrs, on="Origin", how="outer").rename(columns={"Origin": "airport"})

# ---- facts needed later for the data-quality table
DQ_FACTS = dict(rows=len(df), cols_checked=len(COLS8), unique_flight_ids=int(df.flight_id.nunique()),
                splits={s: (str(df[df.split == s].FlightDate.min().date()), str(df[df.split == s].FlightDate.max().date())) for s in SPLITS},
                dq_flags={c: int(df[c].sum()) for c in COLS8 if c.startswith("dq_")})
ONTIME = dict(flights=len(df), operated=int(df.is_operated.sum()), cancel_rate=float(df.Cancelled.mean()), dep_ontime=float(1 - df[df.is_operated].DepDel15.mean()),
              arr_late=float(df[df.is_operated].ArrDel15.mean()), avg_arr_delay=float(df[df.is_operated].ArrDelay.mean()),
              buildup=float(op[op.legno >= 4].DepDelay.mean() - op[op.legno == 1].DepDelay.mean()))
del s_, opd
print("Flight-level tables ready:", ", ".join(k for k in TABLES))

1,641,071 scheduled flights, 1,599,869 operated, 2025-01-01 to 2026-08-31


Flight-level tables ready: gold_dim_date, gold_airport_hour, gold_delay_by_leg


## Building the turnaround table

A **turnaround** is the stop on the ground between two flights of the same aircraft. The buffer question lives here, so every model in this
project works on one row per turnaround. We rebuild the table directly from `aa_clean_v1.0`, so nothing else is needed.

**In plain words, the code below does five things:**
1. Puts each aircraft's flights in time order.
2. Looks at the aircraft's previous flight for every flight.
3. Keeps the pair only if both flights flew, the first landed where the second takes off, they are on the same day, and the planned gap is above zero.
4. Records the **scheduled ground time** (planned gap in minutes), the **inbound delay** `A` (arrival delay of the first flight) and the **outbound delay** `D` (departure delay of the second flight).
5. Adds facts that are known from the timetable alone (hour, airport, distance, how many flights the aircraft has already flown that day). These are the only things the prediction models are allowed to use.

`A` and `D` are limited to the 0.5% and 99.5% levels of the *training* period, so a few extreme flights cannot dominate the averages.

Here we also keep the **flight IDs** of both flights, so that every turnaround has a stable key (`turn_id`) that the dashboard can use.

In [5]:
# limits for extreme delays, learned on the training period only
cap_lo, cap_hi = df.loc[(df.split == "train") & df.ArrDelay.notna(), "ArrDelay"].quantile([0.005, 0.995]).values
dcap_lo, dcap_hi = df.loc[(df.split == "train") & df.DepDelay.notna(), "DepDelay"].quantile([0.005, 0.995]).values
TOP10 = df.Origin.value_counts().head(10).index.tolist()                     # ten busiest departure airports
daily_cancel = df.groupby("FlightDate").Cancelled.mean()
STORM = set(daily_cancel[daily_cancel > 0.10].index)                          # disruption days: more than 10% of flights cancelled
AIRPORTS = sorted(set(df.Origin) | set(df.Dest))

d = df.loc[df.Tail_Number.notna()].sort_values(["Tail_Number", "crs_dep_utc"]).copy()
day = d.groupby(["Tail_Number", "FlightDate"])
d["legno"] = day.cumcount() + 1                                               # which flight of the aircraft's day this is (timetable fact)
d["mins_since_first"] = d.crs_dep_min - day.crs_dep_min.transform("min")      # minutes since the aircraft's first planned departure of the day
g = d.groupby("Tail_Number", sort=False)
d["prev_dest"] = g["Dest"].shift(); d["prev_origin"] = g["Origin"].shift(); d["prev_dist"] = g["Distance"].shift()
d["prev_arr_utc"] = g["crs_arr_utc"].shift(); d["prev_date"] = g["FlightDate"].shift()
d["prev_dep_hr"] = g["crs_dep_min"].shift() // 60; d["prev_arr_hr"] = g["crs_arr_min"].shift() // 60
d["A_raw"] = g["ArrDelay"].shift(); d["prev_flight_id"] = g["flight_id"].shift(); d["prev_flight_no"] = g["Flight_Number_Reporting_Airline"].shift()
d["prev_operated"] = g["is_operated"].shift().eq(True); d["prev_notdiv"] = g["Diverted"].shift().eq(0)
d["sg"] = (d.crs_dep_utc - d.prev_arr_utc).dt.total_seconds() / 60            # scheduled ground time in minutes

link = (d.is_operated & (d.Diverted == 0) & d.prev_operated & d.prev_notdiv & (d.prev_dest == d.Origin)
        & (d.sg > 0) & (d.FlightDate == d.prev_date))
T = d.loc[link, ["Tail_Number", "Origin", "prev_origin", "FlightDate", "split", "Month", "DayOfWeek", "crs_dep_min", "sg", "Distance",
                 "prev_dist", "legno", "mins_since_first", "prev_dep_hr", "prev_arr_hr", "A_raw", "DepDelay",
                 "flight_id", "prev_flight_id", "Flight_Number_Reporting_Airline", "prev_flight_no", "Dest", "crs_dep_utc", "prev_arr_utc"]].rename(
    columns={"Origin": "station", "DepDelay": "D_raw"}).dropna(subset=["A_raw", "D_raw"]).copy()
T["A"] = T.A_raw.clip(cap_lo, cap_hi); T["D"] = T.D_raw.clip(dcap_lo, dcap_hi)
T["hr"] = (T.crs_dep_min // 60).clip(0, 23)
T["hub"] = T.station.isin(TOP10).astype(int)
T["storm"] = T.FlightDate.isin(STORM)
for c in ["station", "prev_origin"]:
    T[c] = pd.Categorical(T[c], categories=AIRPORTS)
T = T.reset_index(drop=True)
del d, g, day, link                                                       # free memory

print(f"Turnarounds built: {len(T):,}   (disruption days: {len(STORM)}; ten busiest airports: {', '.join(TOP10)})")
display(T.groupby("split").agg(turnarounds=("sg", "size"), first_day=("FlightDate", "min"), last_day=("FlightDate", "max"),
                               median_ground_min=("sg", "median"), mean_inbound_delay=("A", "mean"), mean_outbound_delay=("D", "mean")).loc[SPLITS].round(1))
RESULTS["n_turnarounds"] = int(len(T)); RESULTS["n_storm_days"] = len(STORM); RESULTS["top10"] = TOP10

T["turn_id"] = T.prev_flight_id.astype(str) + "__" + T.flight_id.astype(str)
dup = int(T.turn_id.duplicated().sum())
print(f"turn_id is unique: {dup == 0} ({dup} duplicates)")
T = T.drop_duplicates("turn_id").reset_index(drop=True)
T["year_month"] = T.FlightDate.dt.strftime("%Y-%m")
del df, op                     # the flight-level tables are already done; free memory

Turnarounds built: 1,060,740   (disruption days: 27; ten busiest airports: DFW, CLT, MIA, ORD, PHX, PHL, LAX, DCA, MCO, LGA)


,turnarounds,first_day,last_day,median_ground_min,mean_inbound_delay,mean_outbound_delay
split,,,,,,
train,474633,2025-01-01,2025-09-30,61.0,5.9,15.9
validate,160835,2025-10-01,2025-12-31,61.0,4.0,13.6
test,425272,2026-01-01,2026-08-31,60.0,4.8,17.0


turn_id is unique: True (0 duplicates)


In [6]:
def _boot(S, B, seed):
    """Re-draw whole days at random, B times, and add up the stored sums (groups x days x stats)."""
    rng = np.random.default_rng(seed)
    nd = S.shape[1]
    idx = rng.integers(0, nd, size=(B, nd))
    return S[:, idx, :].sum(axis=2)

def slope_table(frame, group, labels, B=200, seed=SEED, min_n=300):
    """Pass-through (slope of outbound delay D on inbound delay A) for each group, with a 95% day-block bootstrap range."""
    f = frame[[group, "FlightDate", "A", "D"]].copy()
    f["AA"] = f.A * f.A; f["AD"] = f.A * f.D; f["n"] = 1.0
    days = np.sort(f.FlightDate.unique()); di = pd.Index(days)
    agg = f.groupby([group, "FlightDate"]).agg(n=("n", "sum"), sA=("A", "sum"), sD=("D", "sum"), sAA=("AA", "sum"), sAD=("AD", "sum")).reset_index()
    S = np.zeros((len(labels), len(days), 5))
    S[agg[group].values.astype(int), di.get_indexer(agg.FlightDate), :] = agg[["n", "sA", "sD", "sAA", "sAD"]].values
    def slope(M):
        n, sA, sD, sAA, sAD = [M[..., k] for k in range(5)]
        with np.errstate(divide="ignore", invalid="ignore"):
            return (n * sAD - sA * sD) / (n * sAA - sA ** 2)
    pt = slope(S.sum(axis=1)); bs = slope(_boot(S, B, seed)) if B > 1 else np.full((len(labels), 2), np.nan)
    out = pd.DataFrame({"group": labels, "n": S[..., 0].sum(axis=1).astype(int), "slope": pt,
                        "lo": np.nanpercentile(bs, 2.5, axis=1) if B > 1 else np.nan,
                        "hi": np.nanpercentile(bs, 97.5, axis=1) if B > 1 else np.nan})
    out.loc[out.n < min_n, ["slope", "lo", "hi"]] = np.nan
    return out

## 4. Pass-through curve (absorption model output)

The same curve as notebook 04: learned on the training period, smoothed so that more ground time can never pass on more delay. We also store the pass-through for
segments (airport type, time of day, disruption, period) in three ground-time bands. **This is the output of the absorption model**, and the buffer engine uses it.

In [7]:
# ---- the pass-through curve, learned on the TRAINING period only ----------------------------------------------------
from sklearn.isotonic import IsotonicRegression

EDGES = [0, 30, 40, 45, 50, 55, 60, 65, 70, 75, 80, 90, 100, 110, 120, 135, 150, 180, 240, 100000]
def edge_label(lo, hi):
    return f"up to {hi}" if lo == 0 else (f"over {lo}" if hi >= 100000 else f"{lo+1}-{hi}")
BIN_LABELS = [edge_label(lo, hi) for lo, hi in zip(EDGES[:-1], EDGES[1:])]

Ptr = T[(T.split == "train") & (T.A > 0)].copy()                 # training turnarounds whose inbound flight was late
Ptr["sgb"] = pd.cut(Ptr.sg, EDGES, labels=False)
raw = slope_table(Ptr, "sgb", BIN_LABELS, B=200)
bin_stats = Ptr.groupby("sgb").agg(sg_mean=("sg", "mean"), A_mean=("A", "mean"), D_mean=("D", "mean")).reindex(range(len(BIN_LABELS)))
raw = pd.concat([raw, bin_stats.reset_index(drop=True)], axis=1)
ok = raw.slope.notna()
# Smooth the curve so that more ground time can never make things worse (a falling line), weighting by group size
iso = IsotonicRegression(increasing=False, y_min=0.0, y_max=1.0, out_of_bounds="clip")
iso.fit(raw.loc[ok, "sg_mean"], raw.loc[ok, "slope"], sample_weight=raw.loc[ok, "n"])
raw["smooth"] = np.nan; raw.loc[ok, "smooth"] = iso.predict(raw.loc[ok, "sg_mean"])

def pass_through(sg):
    """Share of an inbound delay that is still there when the next flight leaves, for a scheduled ground time `sg` (minutes)."""
    x = np.atleast_1d(np.asarray(sg, dtype=float))
    y = iso.predict(x)
    return y if np.ndim(sg) else float(y[0])

In [8]:
rows = []
for r in raw.itertuples():
    rows.append(dict(curve_type="fitted_curve (training period)", segment_type="All turnarounds", segment="All", sg_band=r.group, sg_mean_min=r.sg_mean,
                     turnarounds=int(r.n), pass_through=r.slope, ci95_low=r.lo, ci95_high=r.hi, pass_through_smoothed=r.smooth))
Pall = T[T.A > 0].copy()
Pall["band"] = pd.cut(Pall.sg, [0, 55, 80, 100000], labels=False)
BANDS = ["Short (up to 55)", "Medium (56-80)", "Long (over 80)"]
Pall["tod"] = pd.cut(Pall.hr, [-1, 4, 11, 16, 23], labels=["night", "morning", "afternoon", "evening"])
Pall["all"] = "all"
SPLIT_DEFS = {"All turnarounds": ("all", {"all": "All"}),
              "Airport type": ("hub", {0: "Other airports", 1: "Ten busiest"}),
              "Time of day": ("tod", {"morning": "Morning (05-11)", "afternoon": "Afternoon (12-16)", "evening": "Evening (17-23)"}),
              "Disruption": ("storm", {False: "Normal days", True: "Disruption days"}),
              "Period": ("split", {"train": "Training", "validate": "Validation", "test": "Test"})}
for title, (col, nm) in SPLIT_DEFS.items():
    keys = list(nm); sub = Pall[Pall[col].isin(keys)].copy()
    sub["grp"] = sub[col].map({k: i for i, k in enumerate(keys)}).astype(int) * 3 + sub.band.astype(int)
    tab = slope_table(sub, "grp", [f"{k}|{b}" for k in keys for b in BANDS], B=150)
    bmean = sub.groupby("grp").sg.mean()
    for i, k in enumerate(keys):
        for j, b in enumerate(BANDS):
            r = tab.iloc[i * 3 + j]
            rows.append(dict(curve_type="segment (all periods)", segment_type=title, segment=nm[k], sg_band=b, sg_mean_min=float(bmean.get(i * 3 + j, np.nan)),
                             turnarounds=int(r.n), pass_through=r.slope, ci95_low=r.lo, ci95_high=r.hi, pass_through_smoothed=np.nan))
curve_tab = pd.DataFrame(rows); curve_tab["curve_version"] = CURVE_VERSION
TABLES["gold_passthrough_curve"] = curve_tab
FLAT_UNTIL = float(raw.loc[(raw.smooth >= raw.smooth.max() - 0.03), "sg_mean"].max())
BAND_PT = {b: float(curve_tab[(curve_tab.segment == "All") & (curve_tab.sg_band == b)].pass_through.iloc[0]) for b in BANDS}
print(f"Curve: flat until about {FLAT_UNTIL:.0f} min; pass-through short / medium / long = " + " / ".join(f"{v:.2f}" for v in BAND_PT.values()))
display(curve_tab.head(8).round(3))

Curve: flat until about 63 min; pass-through short / medium / long = 0.98 / 0.97 / 0.28


,curve_type,segment_type,segment,sg_band,sg_mean_min,turnarounds,pass_through,ci95_low,ci95_high,pass_through_smoothed,curve_version
0,fitted_curve (training period),All turnarounds,All,up to 30,16.927,2751,0.553,0.422,0.692,0.993,curve-train2025-1.0
1,fitted_curve (training period),All turnarounds,All,31-40,39.195,7961,0.993,0.972,1.018,0.993,curve-train2025-1.0
2,fitted_curve (training period),All turnarounds,All,41-45,43.585,8691,0.998,0.972,1.027,0.993,curve-train2025-1.0
3,fitted_curve (training period),All turnarounds,All,46-50,49.517,23577,1.005,0.992,1.016,0.993,curve-train2025-1.0
4,fitted_curve (training period),All turnarounds,All,51-55,52.996,20102,1.004,0.977,1.024,0.993,curve-train2025-1.0
5,fitted_curve (training period),All turnarounds,All,56-60,58.630,19053,1.006,0.983,1.026,0.993,curve-train2025-1.0
6,fitted_curve (training period),All turnarounds,All,61-65,62.895,17058,1.015,0.993,1.039,0.993,curve-train2025-1.0
7,fitted_curve (training period),All turnarounds,All,66-70,67.832,12792,0.960,0.912,1.012,0.960,curve-train2025-1.0


## 5. Prediction models (model output)

The same three models as notebook 05, trained on the training period, tuned on validation, and applied to the test period:

* **Late warning** (`p_late15`): chance that the outbound flight leaves 15 or more minutes late.
* **Outbound delay** (`pred_outbound_delay_min`): expected departure delay in minutes.
* **Inbound risk** (`pred_inbound_delay_min`): expected inbound delay (early counted as 0). The buffer engine uses this one.

Only timetable facts are used as inputs; the code stops if a banned input slips in. We also store each model's quality against simple baselines, its risk groups and what it relies on.

In [9]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
FEATS = ["Month", "DayOfWeek", "hr", "sg", "Distance", "prev_dist", "legno", "mins_since_first", "prev_dep_hr", "prev_arr_hr", "station", "prev_origin"]
BANNED = {"A", "D", "A_raw", "D_raw", "storm", "split", "FlightDate", "Tail_Number", "hub", "crs_dep_min", "flight_id", "prev_flight_id", "turn_id"}
assert not (set(FEATS) & BANNED), "A banned input is in the feature list"
T["late"] = (T.D_raw >= 15).astype(int); T["Apos"] = T.A.clip(lower=0)
tr, va, te = [T[T.split == s] for s in SPLITS]
params = dict(n_estimators=1500, learning_rate=0.05, num_leaves=63, min_child_samples=200, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, verbose=-1)
stop = [lgb.early_stopping(50, verbose=False)]
regA = lgb.LGBMRegressor(**params).fit(tr[FEATS], tr.D, eval_set=[(va[FEATS], va.D)], eval_metric="l1", callbacks=stop)
clfA = lgb.LGBMClassifier(**params).fit(tr[FEATS], tr.late, eval_set=[(va[FEATS], va.late)], eval_metric="auc", callbacks=stop)
regB = lgb.LGBMRegressor(**params).fit(tr[FEATS], tr.Apos, eval_set=[(va[FEATS], va.Apos)], eval_metric="l1", callbacks=stop)
TREES = dict(outbound_delay=int(regA.best_iteration_), late_warning=int(clfA.best_iteration_), inbound_risk=int(regB.best_iteration_))
print("Trees used:", TREES)

# ---- baselines (learned on training only), for the metrics table
K = 20
def shrunk_mean(col, frame):
    by_hr = tr.groupby("hr")[col].mean(); m0 = tr[col].mean()
    t_ = tr.groupby(["station", "hr"], observed=True)[col].agg(["sum", "count"])
    h = frame.hr.map(by_hr).fillna(m0).values
    t2 = t_.reindex(pd.MultiIndex.from_arrays([frame.station.astype(str).values, frame.hr.values]))
    return (t2["sum"].fillna(0).values + K * h) / (t2["count"].fillna(0).values + K)
def reg_scores(y, yhat):
    e = np.asarray(y) - np.asarray(yhat)
    return {"MAE": float(np.abs(e).mean()), "RMSE": float(np.sqrt((e ** 2).mean())), "R2": float(1 - (e ** 2).sum() / ((y - np.mean(y)) ** 2).sum())}
mrows = []
for s, F in [("validate", va), ("test", te)]:
    for pred_name, yhat in [("Baseline: training average", np.full(len(F), tr.D.mean())), ("Baseline: airport and hour average", shrunk_mean("D", F)), ("LightGBM model", regA.predict(F[FEATS]))]:
        for k_, v_ in reg_scores(F.D.values, yhat).items(): mrows.append(dict(model="outbound_delay", predictor=pred_name, period=s, measure=k_, value=v_))
    for pred_name, p in [("Baseline: training late-rate", np.full(len(F), tr.late.mean())), ("Baseline: airport and hour late-rate", shrunk_mean("late", F)), ("LightGBM model", clfA.predict_proba(F[FEATS])[:, 1])]:
        mrows += [dict(model="late_warning", predictor=pred_name, period=s, measure="AUC", value=float(roc_auc_score(F.late, p)) if np.std(p) > 0 else 0.5),
                  dict(model="late_warning", predictor=pred_name, period=s, measure="Brier", value=float(brier_score_loss(F.late, p))),
                  dict(model="late_warning", predictor=pred_name, period=s, measure="LogLoss", value=float(log_loss(F.late, np.clip(p, 1e-4, 1 - 1e-4))))]
    for pred_name, yhat in [("Baseline: training average", np.full(len(F), tr.Apos.mean())), ("LightGBM model", regB.predict(F[FEATS]))]:
        for k_, v_ in reg_scores(F.Apos.values, yhat).items(): mrows.append(dict(model="inbound_risk", predictor=pred_name, period=s, measure=k_, value=v_))
        mrows.append(dict(model="inbound_risk", predictor=pred_name, period=s, measure="RankCorrelation",
                          value=float(pd.Series(yhat).rank().corr(pd.Series(F.Apos.values).rank())) if np.std(yhat) > 0 else 0.0))
metrics = pd.DataFrame(mrows); metrics["model_version"] = MODEL_VERSION
TABLES["gold_model_metrics"] = metrics
display(metrics[(metrics.period == "test") & (metrics.predictor == "LightGBM model")].round(3))

Trees used: {'outbound_delay': 252, 'late_warning': 166, 'inbound_risk': 241}


,model,predictor,period,measure,value,model_version
32,outbound_delay,LightGBM model,test,MAE,24.988,lgbm-1.0
33,outbound_delay,LightGBM model,test,RMSE,45.970,lgbm-1.0
34,outbound_delay,LightGBM model,test,R2,0.072,lgbm-1.0
41,late_warning,LightGBM model,test,AUC,0.703,lgbm-1.0
42,late_warning,LightGBM model,test,Brier,0.181,lgbm-1.0
43,late_warning,LightGBM model,test,LogLoss,0.542,lgbm-1.0
48,inbound_risk,LightGBM model,test,MAE,19.370,lgbm-1.0
49,inbound_risk,LightGBM model,test,RMSE,37.553,lgbm-1.0
50,inbound_risk,LightGBM model,test,R2,0.151,lgbm-1.0
51,inbound_risk,LightGBM model,test,RankCorrelation,0.310,lgbm-1.0


In [10]:
# ---- model output for every test-period turnaround
te = te.sort_values(["FlightDate", "crs_dep_utc", "turn_id"]).reset_index(drop=True)
te["p_late15"] = clfA.predict_proba(te[FEATS])[:, 1]
te["pred_outbound_delay_min"] = regA.predict(te[FEATS])
te["pred_inbound_delay_min"] = regB.predict(te[FEATS]).clip(min=0)
te["risk_decile"] = pd.qcut(te.p_late15.rank(method="first"), 10, labels=False) + 1
te["inbound_risk_decile"] = pd.qcut(te.pred_inbound_delay_min.rank(method="first"), 10, labels=False) + 1
te["risk_band"] = pd.cut(te.risk_decile, [0, 3, 7, 10], labels=["Low", "Medium", "High"]).astype(str)

pred = pd.DataFrame({
    "turn_id": te.turn_id, "flight_date": te.FlightDate.dt.date, "year_month": te.year_month, "tail_number": te.Tail_Number, "airport": te.station.astype(str),
    "inbound_from": te.prev_origin.astype(str), "outbound_to": te.Dest, "inbound_flight_id": te.prev_flight_id, "outbound_flight_id": te.flight_id,
    "inbound_flight_no": te.prev_flight_no.astype("Int64"), "outbound_flight_no": te.Flight_Number_Reporting_Airline.astype("Int64"),
    "sched_inbound_arr_utc": te.prev_arr_utc, "sched_outbound_dep_utc": te.crs_dep_utc, "sched_dep_hour_local": te.hr.astype(int),
    "sched_ground_min": te.sg.round(1), "very_short_turn_flag": te.sg < 20, "leg_of_day": te.legno.astype(int), "is_top10_airport": te.hub.astype(bool), "is_disruption_day": te.storm.astype(bool),
    "p_late15": te.p_late15.round(4), "pred_outbound_delay_min": te.pred_outbound_delay_min.round(2), "pred_inbound_delay_min": te.pred_inbound_delay_min.round(2),
    "risk_decile": te.risk_decile.astype(int), "inbound_risk_decile": te.inbound_risk_decile.astype(int), "risk_band": te.risk_band,
    "actual_inbound_delay_min": te.A_raw, "actual_outbound_delay_min": te.D_raw, "actual_late15": te.late.astype(bool),
    "model_version": MODEL_VERSION, "run_id": RUN_ID, "scored_at_utc": RUN_TS.tz_localize(None)})
TABLES["gold_ml_predictions"] = pred
print(f"gold_ml_predictions: {len(pred):,} rows; {int(pred.very_short_turn_flag.sum()):,} have under 20 minutes of planned ground time (flagged: these are usually aircraft swaps or timetable quirks, so treat them with care)")

# ---- risk groups: predicted against observed
dec = pd.concat([
    te.groupby("risk_decile").agg(turnarounds=("late", "size"), predicted=("p_late15", "mean"), observed=("late", "mean")).reset_index().rename(columns={"risk_decile": "decile"}).assign(model="late_warning", unit="share late 15+ min"),
    te.groupby("inbound_risk_decile").agg(turnarounds=("Apos", "size"), predicted=("pred_inbound_delay_min", "mean"), observed=("Apos", "mean")).reset_index().rename(columns={"inbound_risk_decile": "decile"}).assign(model="inbound_risk", unit="inbound delay, minutes")])
dec["period"] = "test"; TABLES["gold_model_deciles"] = dec[["model", "period", "decile", "turnarounds", "predicted", "observed", "unit"]]

# ---- what each model relies on
nice = {"hr": "Planned departure hour", "station": "Airport", "sg": "Planned ground time", "legno": "Flights already flown by the aircraft that day",
        "mins_since_first": "Minutes since the aircraft's first departure", "prev_origin": "Where the arriving flight came from", "Month": "Month", "DayOfWeek": "Weekday",
        "Distance": "Length of next flight", "prev_dist": "Length of arriving flight", "prev_dep_hr": "Arriving flight: planned departure hour", "prev_arr_hr": "Arriving flight: planned arrival hour"}
fi = []
for mname, m in [("outbound_delay", regA), ("late_warning", clfA), ("inbound_risk", regB)]:
    g_ = pd.Series(m.booster_.feature_importance("gain"), index=FEATS); g_ = 100 * g_ / g_.sum()
    for f_, v_ in g_.sort_values(ascending=False).items():
        fi.append(dict(model=mname, feature=f_, feature_label=nice[f_], importance_pct=float(v_)))
fi = pd.DataFrame(fi); fi["rank"] = fi.groupby("model").importance_pct.rank(ascending=False, method="first").astype(int)
TABLES["gold_feature_importance"] = fi

gold_ml_predictions: 425,272 rows; 7,747 have under 20 minutes of planned ground time (flagged: these are usually aircraft swaps or timetable quirks, so treat them with care)


## 6. Buffer engine (optimisation output) and simulator

The same engine and simulator as notebook 06. For each test day and each budget, every strategy spends the same number of extra minutes; the engine puts each block of buffer
where *(predicted inbound delay) x (delay the curve says it absorbs)* is largest.

For the engine we store the **recommendation for every turnaround**, with two views:
* **Before the day (expected):** what the planner would see, based on the predicted inbound delay.
* **After the day (backtest):** what the buffer would really have absorbed, using the real inbound delay.

Each recommendation has a **reason code** in plain words, so a planner can see *why* a turnaround did or did not get buffer.

In [11]:
te["risk"] = te.pred_inbound_delay_min.values      # the engine uses the inbound-risk prediction
BMAX = 30
SIZES = np.arange(0, BMAX + 1, 5)                     # extra minutes a turnaround can get: 0, 5, 10, ... 30
sg, apos, risk, dr = te.sg.values, te.Apos.values, te.risk.values, te.D_raw.values
p0 = pass_through(sg)
P_OPT = np.column_stack([pass_through(sg + o) for o in SIZES])       # pass-through after adding each amount
days, first = np.unique(te.FlightDate.values, return_index=True)
bounds = list(first) + [len(te)]
storm_i = te.storm.values; hub_i = te.hub.values == 1
POLICIES = ["No buffer", "Spread evenly", "Shortest first", "Riskiest first", "Engine (our method)", "Hindsight (not possible)"]

def allocate_day(policy, lo, hi, budget):
    n = hi - lo; b = np.zeros(n)
    if policy == "No buffer" or budget <= 0: return b
    if policy == "Spread evenly":
        return np.full(n, min(budget / n, BMAX))
    if policy in ("Shortest first", "Riskiest first"):
        order = np.argsort(sg[lo:hi], kind="stable") if policy == "Shortest first" else np.argsort(-risk[lo:hi], kind="stable")
        k = int(budget // BMAX); k = min(k, n)
        b[order[:k]] = BMAX
        if k < n: b[order[k]] = min(budget - k * BMAX, BMAX)
        return b
    r = risk[lo:hi] if policy == "Engine (our method)" else apos[lo:hi]
    G = r[:, None] * (p0[lo:hi, None] - P_OPT[lo:hi])               # delay minutes absorbed if the turnaround gets 0, 5, ... 30 extra minutes
    rows_ = np.arange(n)
    # step 1: find the "price per minute" (lam) at which the turnarounds' own best choices just fit the budget
    lam_lo, lam_hi = 0.0, float(G.max() / SIZES[1]) + 1.0
    for _ in range(40):
        lam = (lam_lo + lam_hi) / 2
        if SIZES[(G - lam * SIZES[None, :]).argmax(axis=1)].sum() > budget: lam_lo = lam
        else: lam_hi = lam
    k = (G - lam_hi * SIZES[None, :]).argmax(axis=1); spent = float(SIZES[k].sum())
    # step 2: spend what is left, in rounds, always on the best remaining upgrade
    for _ in range(14):
        dG = G - G[rows_, k][:, None]; dS = SIZES[None, :] - SIZES[k][:, None]
        ratio = np.where(dS > 0, dG / np.where(dS > 0, dS, 1), -np.inf)   # absorbed minutes per extra minute spent, for every possible upgrade
        j = ratio.argmax(axis=1); rr = ratio[rows_, j]
        cand = np.where(rr > 1e-12)[0]
        if len(cand) == 0: break
        order = cand[np.argsort(-rr[cand], kind="stable")]
        cost = (SIZES[j] - SIZES[k])[order]
        take = order[np.cumsum(cost) <= budget - spent]
        if len(take) == 0: break
        spent += cost[:len(take)].sum(); k[take] = j[take]
    b[:] = SIZES[k]
    return b

def run(policy, b0):
    b = np.zeros(len(te))
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        b[lo:hi] = allocate_day(policy, lo, hi, b0 * (hi - lo))
    return b

def evaluate(b, pass_fn=pass_through):
    """Delay minutes absorbed, per turnaround, when each turnaround gets b extra minutes."""
    return apos * (pass_fn(sg) - pass_fn(sg + b))

def summarise(b, absorbed, mask=None):
    m = np.ones(len(te), bool) if mask is None else mask
    base_pass = (apos * p0)[m].sum()
    return dict(turnarounds=int(m.sum()), added_min=float(b[m].sum()), absorbed_min=float(absorbed[m].sum()),
                pct_removed=float(100 * absorbed[m].sum() / base_pass), absorbed_per_100_added=float(100 * absorbed[m].sum() / max(b[m].sum(), 1e-9)),
                late_avoided_per_1000=float(1000 * ((dr[m] >= 15) & (dr[m] - absorbed[m] < 15)).sum() / m.sum()))


SIM = {}
for b0 in BUDGETS:
    for pol in POLICIES:
        b = run(pol, b0); a = evaluate(b)
        SIM[(pol, b0)] = dict(b=b, absorbed=a, daily=np.add.reduceat(a, first), added_daily=np.add.reduceat(b, first))
print("Simulation finished:", len(SIM), "runs over", len(days), "days.")

Simulation finished: 24 runs over 243 days.


In [12]:
# ---- gold_buffer_recommendation: one row per turnaround and budget
G_any = risk[:, None] * (p0[:, None] - P_OPT)              # expected absorbed minutes for 0, 5, ... 30 extra minutes
best_value = (G_any[:, 1:] / SIZES[None, 1:]).max(axis=1)   # best possible expected absorption per buffer minute
recs = []
for b0 in BUDGETS:
    b = SIM[("Engine (our method)", b0)]["b"]; real = SIM[("Engine (our method)", b0)]["absorbed"]
    exp_ = risk * (p0 - pass_through(sg + b))
    vpm = np.where(b > 0, exp_ / np.maximum(b, 1e-9), 0.0)
    reason = np.select(
        [(b > 0) & (sg >= FLAT_UNTIL), (b > 0) & (sg < FLAT_UNTIL), (b == 0) & (best_value < 1e-6), (b == 0) & (te.inbound_risk_decile.values <= 3)],
        ["Added: likely late arrival and ground time already absorbs delay", "Added: enough minutes to reach the range where delay is absorbed",
         "None: too short, extra minutes absorb almost nothing", "None: low inbound risk"],
        default="None: budget used on turnarounds with more value per minute")
    r_ = pd.DataFrame({"turn_id": te.turn_id.values, "flight_date": te.FlightDate.dt.date.values, "year_month": te.year_month.values, "airport": te.station.astype(str).values,
                       "budget_scenario": b0, "current_ground_min": sg.round(1), "recommended_extra_min": b.astype(int), "new_ground_min": (sg + b).round(1),
                       "expected_absorbed_min": exp_.round(3), "value_per_buffer_min": vpm.round(4), "realised_absorbed_min": real.round(3),
                       "late_avoided_flag": (dr >= 15) & (dr - real < 15), "reason_code": reason})
    r_["day_rank"] = r_.assign(v=np.where(b > 0, vpm, np.nan)).groupby("flight_date").v.rank(ascending=False, method="first").astype("Int64")
    recs.append(r_)
rec = pd.concat(recs, ignore_index=True)
rec["engine_version"], rec["curve_version"], rec["run_id"] = ENGINE_VERSION, CURVE_VERSION, RUN_ID
TABLES["gold_buffer_recommendation"] = rec
print(f"gold_buffer_recommendation: {len(rec):,} rows ({len(BUDGETS)} budgets x {len(te):,} turnarounds)")
display(rec[rec.budget_scenario == MAIN_BUDGET].reason_code.value_counts().rename("turnarounds (main budget)").to_frame())

# ---- gold_buffer_plan_daily and gold_buffer_plan_airport
r5 = rec.assign(gets_buffer=rec.recommended_extra_min > 0)
plan_d = r5.groupby(["flight_date", "budget_scenario"]).agg(turnarounds=("turn_id", "size"), turnarounds_with_buffer=("gets_buffer", "sum"),
        buffer_used_min=("recommended_extra_min", "sum"), expected_absorbed_min=("expected_absorbed_min", "sum"),
        realised_absorbed_min=("realised_absorbed_min", "sum"), late_departures_avoided=("late_avoided_flag", "sum")).reset_index()
plan_d["budget_min"] = plan_d.budget_scenario * plan_d.turnarounds
plan_d["is_disruption_day"] = pd.to_datetime(plan_d.flight_date).isin(STORM)
TABLES["gold_buffer_plan_daily"] = plan_d
plan_a = r5.groupby(["airport", "year_month", "budget_scenario"]).agg(turnarounds=("turn_id", "size"), turnarounds_with_buffer=("gets_buffer", "sum"),
        buffer_min=("recommended_extra_min", "sum"), expected_absorbed_min=("expected_absorbed_min", "sum"),
        realised_absorbed_min=("realised_absorbed_min", "sum"), late_departures_avoided=("late_avoided_flag", "sum")).reset_index()
plan_a["share_with_buffer"] = plan_a.turnarounds_with_buffer / plan_a.turnarounds
TABLES["gold_buffer_plan_airport"] = plan_a
del r5

gold_buffer_recommendation: 1,701,088 rows (4 budgets x 425,272 turnarounds)


,turnarounds (main budget)
reason_code,
None: budget used on turnarounds with more value per minute,192685
None: low inbound risk,115244
Added: likely late arrival and ground time already absorbs delay,60886
"None: too short, extra minutes absorb almost nothing",31376
Added: enough minutes to reach the range where delay is absorbed,25081


In [13]:
# ---- gold_sim_strategy: every strategy, budget and day type, with the engine's ratio and its 95% range
rng = np.random.default_rng(SEED); B = 1000
day_storm = pd.Series(days).isin(STORM).values
srows = []
for dtype, mask, dmask in [("all days", np.ones(len(te), bool), np.ones(len(days), bool)), ("normal days", ~storm_i, ~day_storm), ("disruption days", storm_i, day_storm)]:
    nd_ = int(dmask.sum()); idx = rng.integers(0, nd_, size=(B, nd_))
    for b0 in BUDGETS:
        eng_daily = SIM[("Engine (our method)", b0)]["daily"][dmask]
        for pol in POLICIES:
            s = SIM[(pol, b0)]; r = summarise(s["b"], s["absorbed"], mask)
            other = s["daily"][dmask]
            if pol not in ("No buffer", "Engine (our method)") and other.sum() > 0:
                ratios = eng_daily[idx].sum(axis=1) / np.maximum(other[idx].sum(axis=1), 1e-9)
                r.update(engine_vs_this=float(eng_daily.sum() / other.sum()), engine_vs_this_ci95_low=float(np.percentile(ratios, 2.5)), engine_vs_this_ci95_high=float(np.percentile(ratios, 97.5)))
            else:
                r.update(engine_vs_this=np.nan, engine_vs_this_ci95_low=np.nan, engine_vs_this_ci95_high=np.nan)
            srows.append(dict(strategy=pol, budget_scenario=b0, day_type=dtype, **r))
sim_s = pd.DataFrame(srows)
hind = sim_s[sim_s.strategy == "Hindsight (not possible)"].set_index(["budget_scenario", "day_type"]).absorbed_min
sim_s["share_of_hindsight"] = sim_s.absorbed_min / sim_s.set_index(["budget_scenario", "day_type"]).index.map(hind).values
TABLES["gold_sim_strategy"] = sim_s

# ---- gold_sim_daily
sd = []
for (pol, b0), s in SIM.items():
    la = np.add.reduceat(((dr >= 15) & (dr - s["absorbed"] < 15)).astype(float), first)
    sd.append(pd.DataFrame({"flight_date": pd.to_datetime(days).date, "strategy": pol, "budget_scenario": b0, "buffer_added_min": s["added_daily"],
                            "absorbed_min": s["daily"], "late_departures_avoided": la.astype(int), "is_disruption_day": day_storm}))
TABLES["gold_sim_daily"] = pd.concat(sd, ignore_index=True)

# ---- gold_buffer_allocation: where the engine's buffer goes, compared with spreading evenly
band4 = pd.cut(sg, [0, 55, 80, 120, 100000], labels=["Short (up to 55)", "Medium (56-80)", "Long (81-120)", "Very long (over 120)"]).astype(str)
fifth = pd.qcut(pd.Series(risk).rank(method="first"), 5, labels=["1 lowest risk", "2", "3", "4", "5 highest risk"]).astype(str).values
al = []
for b0 in BUDGETS:
    eb, ub = SIM[("Engine (our method)", b0)]["b"], SIM[("Spread evenly", b0)]["b"]
    for gtype, grp in [("Ground-time band", band4), ("Inbound-risk fifth", fifth), ("Airport", te.station.astype(str).values), ("Airport type", np.where(hub_i, "Ten busiest", "Other airports"))]:
        d_ = pd.DataFrame({"group": grp, "engine": eb, "even": ub, "n": 1.0}).groupby("group")[["engine", "even", "n"]].sum()
        for gname, r in d_.iterrows():
            al.append(dict(budget_scenario=b0, grouping=gtype, group=gname, turnarounds=int(r.n), engine_buffer_min=float(r.engine), even_buffer_min=float(r.even),
                           share_of_engine_buffer=float(r.engine / eb.sum()), share_of_turnarounds=float(r.n / len(te))))
TABLES["gold_buffer_allocation"] = pd.DataFrame(al)
display(sim_s[(sim_s.budget_scenario == MAIN_BUDGET) & (sim_s.day_type == "all days")][["strategy", "absorbed_min", "absorbed_per_100_added", "pct_removed", "engine_vs_this", "share_of_hindsight"]].round(2))

,strategy,absorbed_min,absorbed_per_100_added,pct_removed,engine_vs_this,share_of_hindsight
6,No buffer,0.00,0.00,0.00,NaN,0.00
7,Spread evenly,67189.74,3.16,1.32,4.04,0.13
8,Shortest first,40040.80,1.88,0.79,6.77,0.07
9,Riskiest first,223325.58,10.50,4.40,1.21,0.42
10,Engine (our method),271116.55,12.76,5.35,NaN,0.51
11,Hindsight (not possible),536229.24,25.25,10.57,0.51,1.00


## 7. Turnaround columns for the network and airport tables, and the lookups

Now that the turnaround table and the curve exist, we add turnaround measures to the daily network table and to the airport tables.
**Propagated delay** is the inbound delay that the curve says is passed on to the next flight: `inbound delay x pass-through(ground time)`. It is the delay that buffer could target.

In [14]:
T["prop_min"] = T.Apos * pass_through(T.sg.values)
tday = T.groupby("FlightDate").agg(turnarounds=("turn_id", "size"), median_ground_min=("sg", "median"), avg_inbound_delay_min=("A", "mean"),
                                   avg_outbound_delay_min=("D", "mean"), propagated_delay_min=("prop_min", "sum")).reset_index()
net2 = net.merge(tday, on="FlightDate", how="left").rename(columns={"FlightDate": "flight_date"})
net2["is_disruption_day"] = net2.flight_date.isin(STORM)
net2 = net2.merge(dd[["flight_date", "split", "year_month"]], on="flight_date", how="left")
net2["flight_date"] = net2.flight_date.dt.date
TABLES["gold_network_daily"] = net2

# airport x month: add turnaround measures, including pass-through from sums (no bootstrap; groups under 200 late inbounds left empty)
P_ = T[T.A > 0].assign(AA=lambda x: x.A * x.A, AD=lambda x: x.A * x.D, n=1.0)
s_ = P_.groupby(["station", "year_month"], observed=True).agg(n=("n", "sum"), sA=("A", "sum"), sD=("D", "sum"), sAA=("AA", "sum"), sAD=("AD", "sum"))
pt = ((s_.n * s_.sAD - s_.sA * s_.sD) / (s_.n * s_.sAA - s_.sA ** 2)).where(s_.n >= 200)
tm = T.groupby(["station", "year_month"], observed=True).agg(turnarounds=("turn_id", "size"), median_ground_min=("sg", "median"),
        short_turn_share=("sg", lambda x: (x <= 55).mean()), avg_inbound_delay_min=("A", "mean"), avg_outbound_delay_min=("D", "mean"),
        propagated_delay_min=("prop_min", "sum")).reset_index()
tm["pass_through"] = tm.set_index(["station", "year_month"]).index.map(pt).values
tm = tm.rename(columns={"station": "airport"}); tm["airport"] = tm.airport.astype(str)
am2 = am.merge(tm, on=["airport", "year_month"], how="outer")
am2["is_top10_airport"] = am2.airport.isin(TOP10)
TABLES["gold_airport_month"] = am2

# airport lookup
ta = T.groupby("station", observed=True).agg(turnarounds=("turn_id", "size"), median_ground_min=("sg", "median")).reset_index().rename(columns={"station": "airport"})
ta["airport"] = ta.airport.astype(str)
dim_ap2 = dim_ap.merge(ta, on="airport", how="left")
dim_ap2["is_top10"] = dim_ap2.airport.isin(TOP10)
dim_ap2["departure_rank"] = dim_ap2.departures.rank(ascending=False, method="min").astype("Int64")
TABLES["gold_dim_airport"] = dim_ap2.sort_values("departure_rank")
print("Network, airport and lookup tables ready.")

Network, airport and lookup tables ready.


## 8. Headline KPIs

One row per KPI, with its value, unit, period, the page it belongs to, and a definition in plain words. The dashboard's KPI cards read this table directly.

In [15]:
def S(strategy, b0=MAIN_BUDGET, dtype="all days", col="absorbed_min"):
    return float(sim_s[(sim_s.strategy == strategy) & (sim_s.budget_scenario == b0) & (sim_s.day_type == dtype)][col].iloc[0])
def M(model, measure, predictor="LightGBM model", period="test"):
    return float(metrics[(metrics.model == model) & (metrics.measure == measure) & (metrics.predictor == predictor) & (metrics.period == period)].value.iloc[0])
dl = TABLES["gold_model_deciles"]
lw = dl[dl.model == "late_warning"].set_index("decile").observed; ir = dl[dl.model == "inbound_risk"].set_index("decile").observed
E, U = "Engine (our method)", "Spread evenly"
ALLP, TESTP = "Jan 2025 - Aug 2026", "Test: Jan - Aug 2026"
K_ = [
 ("scheduled_flights", ONTIME["flights"], "flights", ALLP, "2 Network", "Scheduled American Airlines mainline flights in the cleaned data"),
 ("operated_flights", ONTIME["operated"], "flights", ALLP, "2 Network", "Flights that were not cancelled"),
 ("cancellation_rate", ONTIME["cancel_rate"], "share", ALLP, "2 Network", "Cancelled flights divided by scheduled flights"),
 ("ontime_departure_rate", ONTIME["dep_ontime"], "share", ALLP, "2 Network", "Operated flights that left less than 15 minutes late"),
 ("late_arrival_rate", ONTIME["arr_late"], "share", ALLP, "2 Network", "Operated flights that arrived 15 or more minutes late"),
 ("avg_arrival_delay", ONTIME["avg_arr_delay"], "minutes", ALLP, "2 Network", "Average arrival delay of operated flights (early arrivals count as negative)"),
 ("disruption_days", len(STORM), "days", ALLP, "2 Network", "Days when more than 10% of scheduled flights were cancelled"),
 ("turnarounds", len(T), "turnarounds", ALLP, "4 Propagation", "Stops between two operated flights of the same aircraft on the same day"),
 ("median_ground_time", float(T.sg.median()), "minutes", ALLP, "4 Propagation", "Median planned ground time at a turnaround"),
 ("short_turn_share", float((T.sg <= 55).mean()), "share", ALLP, "4 Propagation", "Turnarounds with 55 minutes or less of planned ground time"),
 ("pass_through_short", BAND_PT["Short (up to 55)"], "min per min", ALLP, "4 Propagation", "Minutes of outbound delay per minute of inbound delay, ground time up to 55 min"),
 ("pass_through_medium", BAND_PT["Medium (56-80)"], "min per min", ALLP, "4 Propagation", "Same, ground time 56 to 80 min"),
 ("pass_through_long", BAND_PT["Long (over 80)"], "min per min", ALLP, "4 Propagation", "Same, ground time over 80 min"),
 ("absorption_threshold", FLAT_UNTIL, "minutes", "Training: Jan - Sep 2025", "4 Propagation", "Planned ground time below which almost all inbound delay is passed on"),
 ("delay_buildup", ONTIME["buildup"], "minutes", ALLP, "4 Propagation", "Average departure delay of an aircraft's 4th-or-later flight of the day minus its 1st"),
 ("propagated_delay_test", float((te.Apos * p0).sum()), "minutes", TESTP, "4 Propagation", "Inbound delay passed on to the next flight (inbound delay x pass-through), test period"),
 ("late_warning_auc", M("late_warning", "AUC"), "0.5 to 1", TESTP, "5 Risk forecast", "How well the late warning ranks flights (0.5 = coin toss, 1 = perfect)"),
 ("late_warning_auc_baseline", M("late_warning", "AUC", "Baseline: airport and hour late-rate"), "0.5 to 1", TESTP, "5 Risk forecast", "Same, for the simple airport-and-hour baseline"),
 ("top_decile_late_rate", float(lw.loc[10]), "share", TESTP, "5 Risk forecast", "Share late 15+ min among the riskiest tenth of turnarounds"),
 ("bottom_decile_late_rate", float(lw.loc[1]), "share", TESTP, "5 Risk forecast", "Share late 15+ min among the safest tenth"),
 ("late_warning_lift", float(lw.loc[10] / lw.loc[1]), "times", TESTP, "5 Risk forecast", "Riskiest tenth's late rate divided by the safest tenth's"),
 ("inbound_risk_ratio", float(ir.loc[10] / max(ir.loc[1], 1e-9)), "times", TESTP, "5 Risk forecast", "Average inbound delay of the riskiest tenth divided by the safest tenth"),
 ("engine_absorbed", S(E), "minutes", TESTP, "1 Summary / 6 Simulator", f"Delay minutes absorbed by the engine at {MAIN_BUDGET} extra minutes per turnaround"),
 ("engine_efficiency", S(E, col="absorbed_per_100_added"), "per 100 buffer min", TESTP, "1 Summary / 6 Simulator", "Delay minutes absorbed per 100 minutes of buffer (engine)"),
 ("even_efficiency", S(U, col="absorbed_per_100_added"), "per 100 buffer min", TESTP, "6 Simulator", "Same, when the buffer is spread evenly"),
 ("engine_pct_removed", S(E, col="pct_removed") / 100, "share", TESTP, "1 Summary / 6 Simulator", "Share of the propagated delay that the engine's buffer removes"),
 ("engine_late_avoided_per_1000", S(E, col="late_departures_avoided_per_1000") if "late_departures_avoided_per_1000" in sim_s else S(E, col="late_avoided_per_1000"), "per 1,000 turnarounds", TESTP, "1 Summary / 6 Simulator", "Outbound flights moved from 15+ min late to under 15 min late"),
 ("engine_uplift_vs_even", S(U, col="engine_vs_this"), "times", TESTP, "1 Summary / 6 Simulator", "Delay absorbed by the engine divided by delay absorbed when spreading evenly (same budget)"),
 ("engine_uplift_vs_even_ci_low", S(U, col="engine_vs_this_ci95_low"), "times", TESTP, "1 Summary / 6 Simulator", "Lower end of the 95% range of the uplift (re-drawing whole days)"),
 ("engine_uplift_vs_even_ci_high", S(U, col="engine_vs_this_ci95_high"), "times", TESTP, "1 Summary / 6 Simulator", "Upper end of the 95% range of the uplift"),
 ("engine_uplift_vs_riskiest", S("Riskiest first", col="engine_vs_this"), "times", TESTP, "6 Simulator", "Engine divided by 'riskiest first' (same budget)"),
 ("engine_share_of_hindsight", S(E, col="share_of_hindsight"), "share", TESTP, "1 Summary / 6 Simulator", "Engine result divided by the best possible with perfect foresight"),
]
kpi = pd.DataFrame(K_, columns=["kpi_name", "value", "unit", "period", "dashboard_page", "definition"])
kpi["budget_scenario"] = np.where(kpi.kpi_name.str.startswith(("engine", "even")), MAIN_BUDGET, np.nan)
kpi["run_id"] = RUN_ID
TABLES["gold_kpi_headline"] = kpi
display(kpi[["kpi_name", "value", "unit", "period"]].assign(value=lambda x: x.value.round(3)))

,kpi_name,value,unit,period
0,scheduled_flights,1641071.000,flights,Jan 2025 - Aug 2026
1,operated_flights,1599869.000,flights,Jan 2025 - Aug 2026
2,cancellation_rate,0.021,share,Jan 2025 - Aug 2026
3,ontime_departure_rate,0.739,share,Jan 2025 - Aug 2026
4,late_arrival_rate,0.259,share,Jan 2025 - Aug 2026
5,avg_arrival_delay,13.943,minutes,Jan 2025 - Aug 2026
6,disruption_days,27.000,days,Jan 2025 - Aug 2026
7,turnarounds,1060740.000,turnarounds,Jan 2025 - Aug 2026
8,median_ground_time,61.000,minutes,Jan 2025 - Aug 2026
9,short_turn_share,0.381,share,Jan 2025 - Aug 2026


## 9. Data-quality checks and the run log

Every check has an expected value, the actual value and a pass/fail. The **run log** records what produced the tables (data version, periods, model settings, versions),
so every number on the dashboard can be traced back. Checks that compare with notebooks 04 to 06 use a small tolerance, because tiny differences between computers are normal.

In [16]:
chk = []
def add(group, name, expected, actual, ok):
    chk.append(dict(check_group=group, check_name=name, expected=str(expected), actual=str(actual), passed=bool(ok)))
add("Input data", "Checksum of aa_clean_v1.0.parquet", EXPECTED_SHA256[:16] + "...", actual[:16] + "...", actual == EXPECTED_SHA256)
add("Input data", "Rows in aa_clean_v1.0", 1_641_071, DQ_FACTS["rows"], DQ_FACTS["rows"] == 1_641_071)
add("Input data", "flight_id is unique", DQ_FACTS["rows"], DQ_FACTS["unique_flight_ids"], DQ_FACTS["unique_flight_ids"] == DQ_FACTS["rows"])
for s, (a_, b_) in DQ_FACTS["splits"].items():
    add("Input data", f"Period '{s}' dates", {"train": "2025-01-01 to 2025-09-30", "validate": "2025-10-01 to 2025-12-31", "test": "2026-01-01 to 2026-08-31"}[s], f"{a_} to {b_}",
        f"{a_} to {b_}" == {"train": "2025-01-01 to 2025-09-30", "validate": "2025-10-01 to 2025-12-31", "test": "2026-01-01 to 2026-08-31"}[s])
for c, v in DQ_FACTS["dq_flags"].items():
    add("Input data (D2 flags, information)", f"Rows flagged {c}", "information only", v, True)
add("Turnarounds", "turn_id is unique", 0, dup, dup == 0)
add("Turnarounds", "Turnarounds built (notebooks 04-06: 1,060,740)", "1,060,740 +/- 0.1%", f"{len(T):,}", abs(len(T) / 1_060_740 - 1) < 0.001)
add("Models", "No banned input used", "none", ", ".join(sorted(set(FEATS) & BANNED)) or "none", not (set(FEATS) & BANNED))
add("Models", "Late-warning AUC on test (notebook 05: 0.703)", "0.703 +/- 0.01", round(M("late_warning", "AUC"), 4), abs(M("late_warning", "AUC") - 0.703) < 0.01)
add("Models", "Predictions are within valid ranges", "0<=p<=1, inbound>=0", f"p {pred.p_late15.min():.3f}-{pred.p_late15.max():.3f}; inbound min {pred.pred_inbound_delay_min.min():.2f}",
    pred.p_late15.between(0, 1).all() and (pred.pred_inbound_delay_min >= 0).all())
add("Engine", "Daily budget never exceeded", "0 days over", int((plan_d.buffer_used_min > plan_d.budget_min + 1e-6).sum()), (plan_d.buffer_used_min <= plan_d.budget_min + 1e-6).all())
add("Engine", "Recommended minutes between 0 and 30", "0 to 30", f"{rec.recommended_extra_min.min()} to {rec.recommended_extra_min.max()}", rec.recommended_extra_min.between(0, BMAX).all())
add("Engine", "Recommendation table matches simulator total", round(S(E), 1), round(float(rec[rec.budget_scenario == MAIN_BUDGET].realised_absorbed_min.sum()), 1),
    abs(rec[rec.budget_scenario == MAIN_BUDGET].realised_absorbed_min.sum() - S(E)) < 1 + 1e-3 * S(E))
add("Engine", f"Engine vs even spreading at {MAIN_BUDGET} min (notebook 06: 4.04)", "4.04 +/- 0.15", round(S(U, col="engine_vs_this"), 3), abs(S(U, col="engine_vs_this") - 4.04) < 0.15)
for name, t in TABLES.items():
    keys = [c for c in ["turn_id", "flight_date", "airport", "kpi_name", "strategy", "model", "check_name", "decile", "year_month"] if c in t.columns][:2]
    nulls = int(t[keys].isna().sum().sum()) if keys else 0
    add("Output tables", f"{name}: has rows and no missing keys ({', '.join(keys) or 'n/a'})", "rows > 0, 0 missing", f"{len(t):,} rows, {nulls} missing", len(t) > 0 and nulls == 0)
dq = pd.DataFrame(chk); dq["run_id"] = RUN_ID
TABLES["gold_data_quality"] = dq
print(f"Checks passed: {dq.passed.sum()} of {len(dq)}")
display(dq[~dq.passed] if (~dq.passed).any() else Markdown("**All checks passed.**"))

import sklearn
log = pd.DataFrame([dict(run_id=RUN_ID, run_timestamp_utc=RUN_TS.tz_localize(None), environment=ENV, data_tag=DATA_TAG, data_sha256=actual, data_rows=DQ_FACTS["rows"],
    train_period=" to ".join(DQ_FACTS["splits"]["train"]), validate_period=" to ".join(DQ_FACTS["splits"]["validate"]), test_period=" to ".join(DQ_FACTS["splits"]["test"]),
    model_version=MODEL_VERSION, engine_version=ENGINE_VERSION, curve_version=CURVE_VERSION, random_seed=SEED,
    model_settings=json.dumps({k: v for k, v in params.items() if k != "verbose"}), trees_used=json.dumps(TREES),
    budgets=json.dumps(BUDGETS), main_budget=MAIN_BUDGET, max_extra_min_per_turnaround=BMAX, disruption_rule="daily cancellation rate > 10%",
    curve_points=json.dumps({int(s): round(float(pass_through(s)), 4) for s in range(0, 301, 10)}),
    test_late_warning_auc=M("late_warning", "AUC"), test_engine_vs_even=S(U, col="engine_vs_this"), checks_passed=int(dq.passed.sum()), checks_total=len(dq),
    python_version=sys.version.split()[0], pandas_version=pd.__version__, lightgbm_version=lgb.__version__, sklearn_version=sklearn.__version__)])
TABLES["gold_engine_run_log"] = log

Checks passed: 38 of 38


**All checks passed.**

## 10. Table catalogue

A small table that describes every other table (grain, page, row count), so that the frontend team does not need this notebook to understand them.

In [17]:
CAT = {
 "gold_dim_airport": ("airport", "Filters; pages 3, 4, 6", "Airport code, city, state, departures, arrivals, turnarounds, busiest-ten flag, rank"),
 "gold_dim_date": ("day", "Filters; all pages", "Date parts, train/validate/test period, cancellation rate, disruption-day flag"),
 "gold_kpi_headline": ("KPI", "1 Executive summary (and KPI cards elsewhere)", "Every headline number with unit, period and plain-language definition"),
 "gold_network_daily": ("day", "2 Network delay health", "Flights, cancellations, late rates, average delays, turnarounds, propagated delay"),
 "gold_airport_month": ("airport x month", "3 Airport risk", "Departures, late rates, delays, ground time, short-turn share, pass-through"),
 "gold_airport_hour": ("airport x departure hour", "3 Airport risk (heatmap)", "Departures, late rate and average delay by hour"),
 "gold_passthrough_curve": ("ground-time band x segment", "4 Delay propagation", "Pass-through with 95% range; fitted curve and segment view"),
 "gold_delay_by_leg": ("flight number in the aircraft's day x period", "4 Delay propagation", "How delay builds up through the aircraft's day"),
 "gold_ml_predictions": ("turnaround (test period)", "5 Risk forecast; 6a Recommendations", "Model outputs (late probability, outbound and inbound delay, risk groups) with IDs, times and actual outcomes"),
 "gold_model_metrics": ("model x predictor x period x measure", "5 Risk forecast; 7 Trust", "Model quality against baselines on validation and test"),
 "gold_model_deciles": ("model x risk group", "5 Risk forecast", "Predicted against observed in each tenth of risk"),
 "gold_feature_importance": ("model x input", "5 Risk forecast", "What each model relies on"),
 "gold_buffer_recommendation": ("turnaround x budget", "6a Recommendations", "Engine output: recommended extra minutes, expected and realised absorbed delay, reason, rank"),
 "gold_buffer_plan_daily": ("day x budget", "6 Simulator; 6a", "Engine plan per day: budget, minutes used, turnarounds with buffer, expected and realised absorbed"),
 "gold_buffer_plan_airport": ("airport x month x budget", "6 Simulator; 6a", "Engine plan per airport and month"),
 "gold_sim_strategy": ("strategy x budget x day type", "1 Summary; 6 Simulator", "Comparison of strategies, with the engine's ratio and 95% range"),
 "gold_sim_daily": ("day x strategy x budget", "6 Simulator", "Daily absorbed minutes for each strategy"),
 "gold_buffer_allocation": ("budget x grouping x group", "6 Simulator", "Where the engine puts buffer compared with spreading evenly"),
 "gold_data_quality": ("check", "7 Data and model trust", "Checks with expected and actual values and pass/fail"),
 "gold_engine_run_log": ("run", "7 Data and model trust", "Data version, periods, settings and versions behind every table"),
}
catalog = pd.DataFrame([dict(table_name=k, grain=v[0], dashboard_page=v[1], description=v[2], rows=len(TABLES[k]), columns=TABLES[k].shape[1],
                             columns_list=", ".join(TABLES[k].columns)) for k, v in CAT.items()])
catalog["run_id"] = RUN_ID
TABLES["gold_table_catalog"] = catalog
display(catalog[["table_name", "grain", "dashboard_page", "rows", "columns"]])

,table_name,grain,dashboard_page,rows,columns
0,gold_dim_airport,airport,"Filters; pages 3, 4, 6",137,9
1,gold_dim_date,day,Filters; all pages,608,10
2,gold_kpi_headline,KPI,1 Executive summary (and KPI cards elsewhere),32,8
3,gold_network_daily,day,2 Network delay health,608,20
4,gold_airport_month,airport x month,3 Airport risk,2432,17
5,gold_airport_hour,airport x departure hour,3 Airport risk (heatmap),1822,5
6,gold_passthrough_curve,ground-time band x segment,4 Delay propagation,52,11
7,gold_delay_by_leg,flight number in the aircraft's day x period,4 Delay propagation,24,7
8,gold_ml_predictions,turnaround (test period),5 Risk forecast; 6a Recommendations,425272,31
9,gold_model_metrics,model x predictor x period x measure,5 Risk forecast; 7 Trust,52,6


## 11. Save the tables

Every table is saved as a **parquet file** (and as **CSV** if it has at most 300,000 rows). If Spark is available (Databricks), each table is also written as a
**Delta table** `workspace.bufferengine.<table name>`, replacing the previous version. Dates and times are stored in UTC without a time zone, and text categories as plain text, so any tool can read them.

In [18]:
def writable(t):
    w = t.copy()
    for c in w.columns:
        if isinstance(w[c].dtype, pd.CategoricalDtype): w[c] = w[c].astype(str)
        elif isinstance(w[c].dtype, pd.DatetimeTZDtype): w[c] = w[c].dt.tz_convert("UTC").dt.tz_localize(None)
    w.columns = [c.lower() for c in w.columns]
    return w

if WRITE_DELTA:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
saved = []
for name, t in TABLES.items():
    w = writable(t)
    pq_path = PQ_DIR / f"{name}.parquet"
    w.to_parquet(pq_path, index=False, coerce_timestamps="us", allow_truncated_timestamps=True)
    csv_note = "parquet only (too large for CSV)"
    if len(w) <= CSV_MAX_ROWS:
        w.to_csv(CSV_DIR / f"{name}.csv", index=False); csv_note = "parquet + csv"
    delta = "-"
    if WRITE_DELTA:
        spark.read.parquet(str(pq_path)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{name}")
        delta = f"{CATALOG}.{SCHEMA}.{name}"
    saved.append(dict(table=name, rows=len(w), columns=w.shape[1], files=csv_note, delta_table=delta, size_mb=round(pq_path.stat().st_size / 1e6, 2)))
saved = pd.DataFrame(saved)
display(saved)
print(f"Saved {len(saved)} tables in '{OUT.name}'" + (f" and as Delta tables in {CATALOG}.{SCHEMA}." if WRITE_DELTA else "."))

,table,rows,columns,files,delta_table,size_mb
0,gold_dim_date,608,10,parquet + csv,-,0.02
1,gold_airport_hour,1822,5,parquet + csv,-,0.04
2,gold_delay_by_leg,24,7,parquet + csv,-,0.01
3,gold_passthrough_curve,52,11,parquet + csv,-,0.01
4,gold_model_metrics,52,6,parquet + csv,-,0.00
5,gold_ml_predictions,425272,31,parquet only (too large for CSV),-,32.27
6,gold_model_deciles,20,7,parquet + csv,-,0.00
7,gold_feature_importance,36,5,parquet + csv,-,0.00
8,gold_buffer_recommendation,1701088,17,parquet only (too large for CSV),-,52.68
9,gold_buffer_plan_daily,972,10,parquet + csv,-,0.04


Saved 21 tables in 'consumption_tables'.


## 12. Quick look at the most important outputs

A few rows of the two product tables, so you can see what the dashboard will receive.

In [19]:
say("**Model output (`gold_ml_predictions`), five highest-risk turnarounds on the first test day:**")
first_day = pred.flight_date.min()
display(pred[pred.flight_date == first_day].nlargest(5, "p_late15")[["turn_id", "airport", "sched_ground_min", "p_late15", "pred_inbound_delay_min", "risk_band", "actual_late15"]])
say(f"**Engine output (`gold_buffer_recommendation`), budget {MAIN_BUDGET} min, the five turnarounds ranked first on the same day:**")
display(rec[(rec.flight_date == first_day) & (rec.budget_scenario == MAIN_BUDGET) & (rec.day_rank <= 5)].sort_values("day_rank")[
    ["turn_id", "airport", "current_ground_min", "recommended_extra_min", "expected_absorbed_min", "realised_absorbed_min", "reason_code", "day_rank"]])
k_ = kpi.set_index("kpi_name").value
say(f"""## Summary

* **{len(TABLES)} tables** were built and saved{' and written to Delta' if WRITE_DELTA else ''}; **{dq.passed.sum()} of {len(dq)} checks passed**.
* The model output covers **{len(pred):,} test-period turnarounds**; the engine output has **{len(rec):,} rows** ({len(BUDGETS)} budgets).
* Headline numbers match notebooks 04 to 06: late-warning AUC **{k_['late_warning_auc']:.3f}**, engine uplift over even spreading **{k_['engine_uplift_vs_even']:.2f}x** (95% range {k_['engine_uplift_vs_even_ci_low']:.2f} to {k_['engine_uplift_vs_even_ci_high']:.2f}) at {MAIN_BUDGET} min per turnaround.
* **Next step:** when the frontend is chosen, add any extra tables or columns it needs here, and re-run this notebook.

**Reminder of the limits** (shown on the Trust page): the pass-through curve shows association, not cause, so absorbed minutes are upper bounds; buffer has a cost we do not price; and the test period is treated as an upcoming timetable, which is a backtest, not a live forecast.""")

**Model output (`gold_ml_predictions`), five highest-risk turnarounds on the first test day:**

,turn_id,airport,sched_ground_min,p_late15,pred_inbound_delay_min,risk_band,actual_late15
1122,20260101-1100-BNA-DFW-1514__20260101-1583-DFW-ELP-1735,DFW,5.0,0.9661,6.47,High,True
1521,20260101-1149-IND-PHX-1750__20260101-762-PHX-ORD-1950,PHX,1.0,0.9566,3.91,High,True
1477,20260101-2291-PHL-PHX-1553__20260101-2488-PHX-LAS-1909,PHX,5.0,0.9457,3.55,High,True
1560,20260101-3065-STL-MIA-1800__20260101-1219-MIA-ORD-2210,MIA,11.0,0.9456,11.79,High,True
422,20260101-2169-PHX-CLT-0730__20260101-1830-CLT-JAX-1329,CLT,5.0,0.9111,0.98,High,True


**Engine output (`gold_buffer_recommendation`), budget 5 min, the five turnarounds ranked first on the same day:**

,turn_id,airport,current_ground_min,recommended_extra_min,expected_absorbed_min,realised_absorbed_min,reason_code,day_rank
425779,20260101-3020-ELP-DFW-0715__20260101-2740-DFW-RSW-1303,DFW,173.0,30,7.372,14.682,Added: likely late arrival and ground time already absorbs delay,1
425668,20260101-2729-BDL-DFW-0605__20260101-3171-DFW-BZN-1221,DFW,183.0,30,7.109,8.697,Added: likely late arrival and ground time already absorbs delay,2
426302,20260101-2390-ATL-DFW-1214__20260101-1366-DFW-BOS-1655,DFW,175.0,30,6.601,0.000,Added: likely late arrival and ground time already absorbs delay,3
426831,20260101-2932-MIA-DFW-1530__20260101-3215-DFW-MCI-2110,DFW,189.0,30,5.734,2.240,Added: likely late arrival and ground time already absorbs delay,4
426223,20260101-1264-TPA-LGA-1135__20260101-1084-LGA-CLT-1730,LGA,194.0,15,2.498,0.000,Added: likely late arrival and ground time already absorbs delay,5


## Summary

* **21 tables** were built and saved; **38 of 38 checks passed**.
* The model output covers **425,272 test-period turnarounds**; the engine output has **1,701,088 rows** (4 budgets).
* Headline numbers match notebooks 04 to 06: late-warning AUC **0.703**, engine uplift over even spreading **4.04x** (95% range 3.97 to 4.11) at 5 min per turnaround.
* **Next step:** when the frontend is chosen, add any extra tables or columns it needs here, and re-run this notebook.

**Reminder of the limits** (shown on the Trust page): the pass-through curve shows association, not cause, so absorbed minutes are upper bounds; buffer has a cost we do not price; and the test period is treated as an upcoming timetable, which is a backtest, not a live forecast.